# Đăng Trần – Step 3, 4, 5: Tokenization, Stop word, Stemming/Lemmatization
Dự án Music_SEG (tìm kiếm lời bài hát). Văn bản tìm kiếm = `title + artist + lyrics`, dùng `clean_text` của Thành.
Cuối notebook có `preprocess_tokens(text, strategy)` gửi Quý.

In [1]:
import os, sqlite3, re, html, unicodedata, random
import pandas as pd
from collections import Counter
random.seed(0)

DB_PATH = os.environ.get("SONGS_DB", "songs.db")   # file .db KHÔNG push lên GitHub
con = sqlite3.connect(DB_PATH)
df = pd.read_sql("SELECT * FROM songs", con)
con.close()
print(df.shape, list(df.columns))

(2765, 8) ['id', 'url', 'title', 'artist', 'album', 'genre', 'lyrics', 'crawled_at']


> Giả định cột `id, title, artist, lyrics, url` (sửa tên cột ở đây nếu khác).

In [2]:
from urllib.parse import urlparse
from Thanh_clean_text import clean_text      # bản chính thức của Thành (file Thanh_clean_text.py cùng thư mục)

df["site"] = df["url"].map(lambda u: urlparse(u).netloc.replace("www.", ""))
df["category"] = df["site"]
# văn bản tìm kiếm = title + artist + lyrics ; rác từng web chỉ xoá trên phần lyrics (clean_text(text, site))
df["lyrics_clean"] = [clean_text(t, s) for t, s in zip(df["lyrics"], df["site"])]
df["text_raw"] = df["title"].fillna("") + " . " + df["artist"].fillna("") + " . " + df["lyrics"].fillna("")
df["text_clean"] = (df["title"].map(clean_text) + " " + df["artist"].map(clean_text) + " " + df["lyrics_clean"]).str.strip()
n_all = len(df)
df = df[df["lyrics_clean"].str.len() > 0].reset_index(drop=True)      # corpus như Thành: chỉ bài có lời thật
print(f"Corpus: {len(df)} / {n_all} bài có lời | theo web:", df.site.value_counts().to_dict())
df[["id","site","title","text_clean"]].head(2)

clean_text OK
Corpus: 2130 / 2765 bài có lời | theo web: {'open.spotify.com': 1566, 'nhaccuatui.com': 315, 'hopamchuan.com': 227, 'nhac.vn': 22}


,id,site,title,text_clean
0,1,nhaccuatui.com,SỚM MUỘN THÌ,"sớm muộn thì anh trai ""say hi"", hustlang robbe..."
1,2,nhaccuatui.com,Pin Dự Phòng,"pin dự phòng dương domic, lou hoàng có những l..."


In [3]:
# ===== Dang_preprocess_tokens: tokenize / stopword / stem / lemma / bỏ dấu =====
import re, unicodedata
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer

def _ensure(pkg, path):
    try:
        nltk.data.find(path)
    except LookupError:
        try: nltk.download(pkg, quiet=True)
        except Exception as e: print("Không tải được", pkg, e)

for _p, _path in [("punkt","tokenizers/punkt"), ("punkt_tab","tokenizers/punkt_tab"),
                  ("stopwords","corpora/stopwords"), ("wordnet","corpora/wordnet"),
                  ("omw-1.4","corpora/omw-1.4")]:
    _ensure(_p, _path)

def tokenize(text):
    """word_tokenize; nếu thiếu punkt thì dùng NLTKWordTokenizer (cùng bộ tách từ, không tách câu)."""
    try:
        from nltk.tokenize import word_tokenize
        return word_tokenize(text)
    except LookupError:
        from nltk.tokenize import NLTKWordTokenizer
        return NLTKWordTokenizer().tokenize(text)

# ---- Từ phủ định: KHÔNG bao giờ bỏ ----
NEGATIONS = {"không","khong","chẳng","chang","chưa","chua","đừng","dung","chớ","chả",
             "not","no","never","nor","n't","nt","cannot","without","none","nothing","nobody","nowhere"}

def _is_neg(w):
    return w in NEGATIONS or w.endswith("n't") or w.endswith("nt") and w[:-2] in {
        "do","does","did","is","are","was","were","have","has","had","would","could","should","can","won","wo","ai","need","must","might","ca","sha"}

def build_en_stopwords():
    try:
        from nltk.corpus import stopwords
        sw = set(stopwords.words("english"))
    except LookupError:
        sw = set()
    # chuẩn hoá giống clean_text (bỏ dấu nháy: don't -> dont) rồi loại hết dạng phủ định
    sw |= {w.replace("'", "") for w in sw}
    return {w for w in sw if not _is_neg(w)}

# Hư từ tiếng Việt (tự soạn, dựa trên các token xuất hiện nhiều nhất của kho — xem Step 4).
# Đại từ nhân xưng (anh, em, tôi, mình...) KHÔNG đưa vào mặc định vì mang nghĩa trong lời nhạc.
VI_STOPWORDS = {
    "và","là","của","thì","những","đã","sẽ","này","các","một","có","cho","với","trong","được",
    "để","đang","khi","mà","cũng","như","nhưng","rằng","vì","nên","ở","ra","vào","lại","nữa",
    "đó","kia","ấy","nào","rồi","còn","hay","hoặc","cả","mọi","từng","bởi","tại","theo","từ",
    "đến","lên","xuống","thế","vẫn","cứ","chỉ","đều","rất","quá","thật","sao","nhé","nha","ơi",
    "à","ừ","ạ","hả","thôi","luôn","mỗi","bao","nhiêu","gì","đâu","ai","vậy","thế","đây",
}
VI_STOPWORDS -= NEGATIONS

def build_stopwords():
    return (build_en_stopwords() | VI_STOPWORDS) - NEGATIONS

STOPWORDS = build_stopwords()
_stemmer, _lemmatizer = PorterStemmer(), WordNetLemmatizer()

def strip_diacritics(s):
    """lạc trôi -> lac troi ; đ -> d"""
    s = unicodedata.normalize("NFD", s.replace("đ","d").replace("Đ","D"))
    return unicodedata.normalize("NFC", "".join(c for c in s if unicodedata.category(c) != "Mn"))

def _wn_pos(tag):
    from nltk.corpus import wordnet
    return {"J": wordnet.ADJ, "V": wordnet.VERB, "N": wordnet.NOUN, "R": wordnet.ADV}.get(tag[:1], wordnet.NOUN)

def lemmatize_tokens(tokens):
    """Như PDF: pos_tag rồi WordNetLemmatizer. Nếu thiếu tagger/wordnet thì lùi về thử n/v, lấy dạng ngắn hơn."""
    try:
        return [_lemmatizer.lemmatize(t, _wn_pos(tag)) for t, tag in nltk.pos_tag(tokens)]
    except LookupError:
        out = []
        for t in tokens:
            try: out.append(min(_lemmatizer.lemmatize(t, "n"), _lemmatizer.lemmatize(t, "v"), key=len))
            except LookupError: out.append(t)
        return out

def preprocess_tokens(text, strategy="baseline"):
    """text: chuỗi ĐÃ qua clean_text. Dùng CHUNG cho bài hát và query.
    baseline      : tokenize + giữ token chữ/số (isalnum), như tokenize_for_search trong PDF
    no_stopwords  : baseline + bỏ stop word (EN + VI, giữ phủ định)
    stemmed       : no_stopwords + PorterStemmer          (như PDF Step 7)
    lemmatized    : no_stopwords + WordNetLemmatizer(POS) (như PDF Step 7)
    no_diacritics : baseline + bỏ dấu (lạc trôi -> lac troi)
    Có thể ghép bằng '+', vd 'no_stopwords+no_diacritics'."""
    parts = set(strategy.split("+")) - {"baseline"}
    bad = parts - {"no_stopwords","stemmed","lemmatized","no_diacritics"}
    if bad: raise ValueError(f"strategy không hợp lệ: {bad}")
    if parts & {"stemmed","lemmatized"}: parts.add("no_stopwords")
    text = text.replace("'", "").replace("\u2019", "")                     # don't -> dont
    text = re.sub(r"[^\w\s]|_", " ", text)                                   # dấu câu, gạch nối -> khoảng trắng (m-tp -> m tp); giữ chữ có dấu
    toks = [t for t in tokenize(text) if t.isalnum()]
    if "no_stopwords" in parts:  toks = [t for t in toks if t not in STOPWORDS]
    if "lemmatized" in parts:    toks = lemmatize_tokens(toks)
    if "stemmed" in parts:       toks = [_stemmer.stem(t) for t in toks]
    if "no_diacritics" in parts: toks = [strip_diacritics(t) for t in toks]
    return toks

## Step 3 – Tokenization (`word_tokenize` trên `clean_text`)

In [4]:
def vi_ratio(s):  # tỉ lệ chữ có dấu tiếng Việt
    letters = [c for c in s if c.isalpha()]
    return sum(1 for c in letters if unicodedata.normalize("NFD", c) != c) / max(len(letters), 1)
def en_ratio(s):
    letters = [c for c in s if c.isalpha()]
    return sum(1 for c in letters if c.isascii()) / max(len(letters), 1)

d = df[df.text_clean.str.split().str.len() > 100].copy()
d["vi"] = d.text_clean.map(vi_ratio)
viet  = d[d.vi > 0.25].sample(1, random_state=1).iloc[0]                    # thuần Việt
eng   = d[d.vi < 0.005].sample(1, random_state=1).iloc[0]                   # thuần Anh
mixed = d[(d.vi.between(0.03, 0.25)) & (d.vi < d.vi.max())].sample(1, random_state=1).iloc[0]      # trộn
samples = {"Việt": viet, "Anh": eng, "Trộn": mixed}

rows = []
for name, r in samples.items():
    tok_raw, tok_clean = tokenize(r.text_raw), tokenize(r.text_clean)
    print(f"[{name}] id={r.id} | {r.title} – {r.artist}")
    print("  30 token đầu:", tok_clean[:30], "\n")
    rows.append({"loại": name, "id": r.id, "token_raw": len(tok_raw), "token_clean": len(tok_clean),
                 "giảm": f"{1-len(tok_clean)/len(tok_raw):.1%}"})
pd.DataFrame(rows)

[Việt] id=908 | Rời Bỏ – Hoà Minzy
  30 token đầu: ['rời', 'bỏ', 'hoà', 'minzy', 'hết', 'hôm', 'nay', 'là', 'em', 'đi', 'khỏi', 'nơi', 'đây', 'anh', 'sẽ', 'không', 'phải', 'nhìn', 'thấy', 'em', 'nữa', 'đâu', 'khoảnh', 'khắc', 'đắm', 'chìm', 'vào', 'nụ', 'cười', 'ấy'] 

[Anh] id=1173 | Please Don't Change (feat. DJ Snake) – Jung Kook, DJ Snake
  30 token đầu: ['please', 'do', "n't", 'change', '(', 'feat', '.', 'dj', 'snake', ')', 'jung', 'kook', ',', 'dj', 'snake', 'lights', 'on', 'camera', 'fast', 'life', 'stamina', 'nowhere', 'to', 'hide', 'on', 'a', 'satellite', 'life', 'on', 'camera'] 

[Trộn] id=82 | NÉT – Cường Seven, Phan Đinh Tùng, Bằng Kiều, Jun Phạm, Tự Long, Trương Thế Vinh, Tuấn Hưng, SOOBIN, S.T Sơn Thạch, Kay Trần, BB Trần, (S)TRONG, Kiên Ứng
  30 token đầu: ['nét', 'cường', 'seven', ',', 'phan', 'đinh', 'tùng', ',', 'bằng', 'kiều', ',', 'jun', 'phạm', ',', 'tự', 'long', ',', 'trương', 'thế', 'vinh', ',', 'tuấn', 'hưng', ',', 'soobin', ',', 's.t', 'sơn', 'thạch', ','] 



,loại,id,token_raw,token_clean,giảm
0,Việt,908,320,315,1.6%
1,Anh,1173,220,218,0.9%
2,Trộn,82,638,605,5.2%


**Nhận xét Step 3.**

- Sau `clean_text`, số token chỉ giảm nhẹ: bài Việt 320 → 315 (1.6%), bài Anh 220 → 218 (0.9%), bài trộn 638 → 605 (5.2%). Lý do: `clean_text` chỉ bỏ nhãn đoạn ([Verse], ĐK:), URL và rác riêng từng website, còn **giữ dấu câu**, nên dấu câu vẫn là token riêng cho đến khi lọc chữ/số ở Step 4. Bài trộn giảm nhiều hơn, nhiều khả năng vì có nhiều nhãn người hát/đoạn hơn.
- **Tiếng Việt bị tách theo âm tiết**: `word_tokenize` chỉ tách theo khoảng trắng và dấu câu (luật dành cho tiếng Anh). Tiếng Việt viết mỗi âm tiết cách nhau một dấu cách nên "tình yêu" → `tình`, `yêu`; "mùa đông" → `mùa`, `đông`. Từ ghép mất ranh giới; muốn giữ phải dùng bộ tách từ tiếng Việt (underthesea, pyvi) hoặc bigram.
- Với tìm kiếm vẫn dùng được vì query cũng được tách giống hệt bài hát; chỉ mất khả năng phân biệt cụm từ.

## Step 4 – Stop word (NLTK EN + hư từ tiếng Việt tự soạn), KHÔNG bỏ phủ định

In [5]:
df["tok"] = df.text_clean.map(tokenize)
cnt = Counter(t for ts in df.tok for t in ts)
print("Top 60 token (để rà soát tay list hư từ tiếng Việt):")
print(cnt.most_common(60))

Top 60 token (để rà soát tay list hư từ tiếng Việt):
[('em', 24679), (',', 24532), ('anh', 22006), ('không', 9823), ('là', 8468), ('yêu', 7859), ('đi', 7626), ('có', 7472), ('ta', 7422), (')', 7342), ('(', 7341), ('một', 7095), ('người', 6330), ('như', 5976), ('ngày', 5562), ('đã', 5559), ('cho', 5323), ('trong', 5275), ('những', 5208), ('mình', 4977), ('tình', 4779), ('về', 4740), ('khi', 4708), ('nhau', 4617), ('còn', 4603), ('ai', 4529), ('để', 4431), ('lại', 4250), ('và', 4191), ('xa', 4106), ('bao', 4012), ('sẽ', 4011), ('rồi', 3896), ('sao', 3850), ('chẳng', 3734), ('bên', 3643), ('biết', 3642), ('?', 3628), ('vẫn', 3618), ('mà', 3543), ('vì', 3528), ('chỉ', 3504), ('đến', 3502), ('đâu', 3499), ('này', 3281), ('đôi', 3266), ('đây', 3250), ('you', 3191), ('nhớ', 3183), ('i', 3122), ('giờ', 3101), ('tay', 3020), ('qua', 3013), ('tim', 3012), ('tôi', 2983), ('con', 2946), ('thương', 2926), ('nói', 2919), ('nơi', 2916), ('đêm', 2841)]


In [6]:
en_sw = build_en_stopwords()
print("EN stopwords:", len(en_sw), "| VI stopwords:", len(VI_STOPWORDS), "| tổng:", len(STOPWORDS))
print("Còn phủ định trong stoplist?", sorted(w for w in STOPWORDS if w in NEGATIONS or w.endswith("nt")))
print("Phủ định giữ lại (có trong kho):", {w: cnt[w] for w in ["không","chẳng","chưa","đừng","not","no","never"] if w in cnt})

EN stopwords: 202 | VI stopwords: 74 | tổng: 276
Còn phủ định trong stoplist? []
Phủ định giữ lại (có trong kho): {'không': 9823, 'chẳng': 3734, 'chưa': 1361, 'đừng': 1952, 'not': 143, 'no': 645, 'never': 182}


In [7]:
# Như PDF: giữ danh sách gốc để so sánh
clean_tokens = [tokenize(t) for t in df.text_clean]                       # sau clean_text
tok_A = [[t for t in ts if t not in STOPWORDS] for ts in clean_tokens]     # Experiment A: chỉ bỏ stop word
tok_B = [[t for t in ts if t.isalnum() and t not in STOPWORDS] for ts in clean_tokens]  # B: bỏ stop word + token không phải chữ/số
n0, nA, nB = (sum(map(len, x)) for x in (clean_tokens, tok_A, tok_B))
print(f"Gốc (sau clean): {n0:,} token")
print(f"A (bỏ stop word)          : {nA:,} token | giảm {(n0-nA)/n0*100:.2f}%")
print(f"B (+ chỉ giữ chữ/số)      : {nB:,} token | giảm {(n0-nB)/n0*100:.2f}%")
print(f"Riêng phần lọc non-alnum (A→B): {nA-nB:,} token")
i = next(k for k, ts in enumerate(clean_tokens) if len(ts) > 40)
print("\nGốc:", clean_tokens[i][:30]); print("A  :", tok_A[i][:30]); print("B  :", tok_B[i][:30])

Gốc (sau clean): 880,654 token
A (bỏ stop word)          : 664,061 token | giảm 24.59%
B (+ chỉ giữ chữ/số)      : 607,390 token | giảm 31.03%
Riêng phần lọc non-alnum (A→B): 56,671 token

Gốc: ['sớm', 'muộn', 'thì', 'anh', 'trai', '``', 'say', 'hi', "''", ',', 'hustlang', 'robber', ',', 'nhâm', 'phương', 'nam', ',', 'mason', 'nguyen', ',', 'jaysonlei', ',', 'khoi', 'vu', ',', 'lamoon', 'baby', 'à', 'anh', 'biết']
A  : ['sớm', 'muộn', 'anh', 'trai', '``', 'say', 'hi', "''", ',', 'hustlang', 'robber', ',', 'nhâm', 'phương', 'nam', ',', 'mason', 'nguyen', ',', 'jaysonlei', ',', 'khoi', 'vu', ',', 'lamoon', 'baby', 'anh', 'biết', 'sớm', 'muộn']
B  : ['sớm', 'muộn', 'anh', 'trai', 'say', 'hi', 'hustlang', 'robber', 'nhâm', 'phương', 'nam', 'mason', 'nguyen', 'jaysonlei', 'khoi', 'vu', 'lamoon', 'baby', 'anh', 'biết', 'sớm', 'muộn', 'mình', 'quay', 'trở', 'về', 'bên', 'nhau', 'phải', 'không']


**Kết quả Step 4.** Cả kho có 880,654 token (sau clean_text). Experiment A (bỏ stop word) còn 664,061 token, giảm **24.59%**. Experiment B (bỏ stop word và chỉ giữ token chữ/số) còn 607,390, giảm **31.03%**: phần lọc dấu câu bỏ thêm 56,671 token (≈6.4%). Phủ định (không, chẳng, chưa, đừng, no, never) được giữ lại đầy đủ.

**Thảo luận – bỏ stop word có luôn tốt hơn?** Không. Stop word chiếm nhiều token nhưng rất ít từ khác nhau: token giảm ~25% mà vocab chỉ giảm 2.2% (9,708 → 9,496). Bỏ chúng thu nhỏ chỉ mục nhưng (1) IDF đã tự hạ trọng số từ phổ biến nên lợi ích với xếp hạng không lớn; (2) query dạng một câu hát ("em là của anh") có thể chỉ gồm hư từ, bỏ đi thì query rỗng; (3) từ phủ định đổi nghĩa câu nên phải giữ. Quyết định dựa trên P@5/R@5 ở Step 7 của Quý.

## Step 5 – Stemming vs Lemmatization + bỏ dấu

In [8]:
sample_words = ["connect","connected","connecting","connection","studies","studying","recorders","recording"]
for w in sample_words:
    print(f"{w:15} -> stem: {_stemmer.stem(w)}")
print()
for w in ["cars","studies","running","better"]:
    print(w, "-> noun:", _lemmatizer.lemmatize(w, pos="n"), "| verb:", _lemmatizer.lemmatize(w, pos="v"), "| adjective:", _lemmatizer.lemmatize(w, pos="a"))
print("\nstem('ngày') =", repr(_stemmer.stem("ngày")), "| lemma('ngày') =", repr(_lemmatizer.lemmatize("ngày")))
for w in ["lạc","trôi","tình","yêu","mùa","đông","nhớ"]:
    print(w, "→ stem:", _stemmer.stem(w), "| lemma:", _lemmatizer.lemmatize(w), "| bỏ dấu:", strip_diacritics(w))
print(strip_diacritics("lạc trôi"))

connect         -> stem: connect
connected       -> stem: connect
connecting      -> stem: connect
connection      -> stem: connect
studies         -> stem: studi
studying        -> stem: studi
recorders       -> stem: record
recording       -> stem: record

cars -> noun: car | verb: cars | adjective: cars
studies -> noun: study | verb: study | adjective: studies
running -> noun: running | verb: run | adjective: running
better -> noun: better | verb: better | adjective: good

stem('ngày') = 'ngài' | lemma('ngày') = 'ngày'
lạc → stem: lạc | lemma: lạc | bỏ dấu: lac
trôi → stem: trôi | lemma: trôi | bỏ dấu: troi
tình → stem: tình | lemma: tình | bỏ dấu: tinh
yêu → stem: yêu | lemma: yêu | bỏ dấu: yeu
mùa → stem: mùa | lemma: mùa | bỏ dấu: mua
đông → stem: đông | lemma: đông | bỏ dấu: dong
nhớ → stem: nhớ | lemma: nhớ | bỏ dấu: nho
lac troi


In [9]:
# Số term khác nhau trên cả kho (stem/lemma tính trên tokens đã bỏ stop word, như PDF)
reps = {s: [preprocess_tokens(t, s) for t in df.text_clean]
        for s in ["baseline","no_stopwords","stemmed","lemmatized","no_diacritics"]}
vocab_sizes = {s: len({t for ts in v for t in ts}) for s, v in reps.items()}
tokens_total = {s: sum(map(len, v)) for s, v in reps.items()}
base = tokens_total["baseline"]
for s in reps:
    print(f"{s:14} vocab = {vocab_sizes[s]:>7,} | tổng token = {tokens_total[s]:>9,} | giảm token so với baseline = {(base-tokens_total[s])/base*100:5.2f}%")
vocab = {t for ts in reps["baseline"] for t in ts}
A = reps["baseline"]

baseline       vocab =   9,708 | tổng token =   836,711 | giảm token so với baseline =  0.00%
no_stopwords   vocab =   9,496 | tổng token =   619,884 | giảm token so với baseline = 25.91%
stemmed        vocab =   8,986 | tổng token =   619,884 | giảm token so với baseline = 25.91%
lemmatized     vocab =   9,115 | tổng token =   619,884 | giảm token so với baseline = 25.91%
no_diacritics  vocab =   6,660 | tổng token =   836,711 | giảm token so với baseline =  0.00%


**Nhận xét Step 5.** Số term khác nhau trên cả kho (stem/lemma tính sau khi bỏ stop word): baseline 9,708; bỏ stop word 9,496; stemming 8,986 (−7.4% so với baseline); lemmatization 9,115 (−6.1%); bỏ dấu 6,660 (−31.4%). Số token: baseline 836,711, các cách bỏ stop word còn 619,884 (giảm 25.91%; mức này khác 24.59% ở Step 4 vì baseline ở đây đã bỏ dấu câu).

- `stemmer.stem("ngày")` ra `ngài`: Porter chỉ có luật tiếng Anh (luật y→i cuối từ biến "ngày" thành "ngài", một từ khác nghĩa). WordNet chỉ có từ điển tiếng Anh nên lemmatizer để nguyên. Tiếng Việt không biến hình nên không có hậu tố nào cần cắt; vì vậy hai cách này gần như chỉ ảnh hưởng phần tiếng Anh (vocab giảm chỉ ~4–5% so với bước bỏ stop word), và với tiếng Việt stemmer chỉ có thể gộp nhầm.
- **Stemming** cắt hậu tố theo luật, nhanh nhưng có thể ra non-word (studies → studi) và gộp quá tay, nên vocab giảm nhiều hơn lemma (8,986 so với 9,115). **Lemmatization** tra từ điển và từ loại, ra từ thật (studies → study, better → good khi pos='a'), chậm hơn, mặc định pos='n' nên cần POS tag mới đúng cho động từ.
- **Bỏ dấu** (lạc trôi → lac troi) giảm vocab mạnh nhất vì gộp nhiều từ khác nghĩa (ma, má, mà, mả, mã, mạ → ma). Có ích với người gõ không dấu và phải áp dụng đồng thời cho bài hát và query, nhưng làm mất phân biệt nghĩa và có thể tăng nhiễu.

**Bigram.** Unigram mất thứ tự: "prime minister" thành `prime`, `minister` riêng lẻ; bigram giữ cụm và tăng độ đặc hiệu. Tiếng Việt cũng vậy: `tình yêu` (1,633 lần), `mùa đông` (282 lần) là một đơn vị nghĩa, nhưng `tình` và `yêu` rời rạc rất phổ biến. Đổi lại số bigram khác nhau là 155,021 so với 9,708 unigram (gấp ~16 lần), ma trận thưa hơn.

In [10]:
from nltk import bigrams
bg = Counter(b for x in A for b in bigrams(x))
print([(" ".join(b), c) for b, c in bg.most_common(25)])
print("Bigram 'tình yêu':", bg[("tình","yêu")], "| 'mùa đông':", bg[("mùa","đông")])
print("Số bigram khác nhau:", f"{len(bg):,}", "so với unigram:", f"{len(vocab):,}")

[('tình yêu', 1633), ('la la', 1550), ('oh oh', 1481), ('ah ah', 1239), ('giấc mơ', 1162), ('yêu em', 1085), ('trái tim', 1060), ('yêu thương', 966), ('cho anh', 923), ('anh không', 922), ('bao nhiêu', 914), ('em đã', 907), ('cho em', 844), ('em không', 837), ('anh đã', 819), ('con tim', 810), ('không còn', 769), ('thời gian', 759), ('chúng ta', 755), ('uh uh', 723), ('bên nhau', 706), ('cô đơn', 686), ('anh vẫn', 672), ('nước mắt', 657), ('một người', 642)]
Bigram 'tình yêu': 1633 | 'mùa đông': 282
Số bigram khác nhau: 155,021 so với unigram: 9,708


## `preprocess_tokens(text, strategy)` – gửi Quý

In [11]:
demo = clean_text("Lạc trôi – Sơn Tùng M-TP. Don't stop loving the dancing hearts, không chẳng bao giờ!")
for s in ["baseline","no_stopwords","stemmed","lemmatized","no_diacritics","no_stopwords+no_diacritics"]:
    print(f"{s:28}", preprocess_tokens(demo, s))

baseline                     ['lạc', 'trôi', 'sơn', 'tùng', 'm', 'tp', 'dont', 'stop', 'loving', 'the', 'dancing', 'hearts', 'không', 'chẳng', 'bao', 'giờ']
no_stopwords                 ['lạc', 'trôi', 'sơn', 'tùng', 'tp', 'dont', 'stop', 'loving', 'dancing', 'hearts', 'không', 'chẳng', 'giờ']
stemmed                      ['lạc', 'trôi', 'sơn', 'tùng', 'tp', 'dont', 'stop', 'love', 'danc', 'heart', 'không', 'chẳng', 'giờ']
lemmatized                   ['lạc', 'trôi', 'sơn', 'tùng', 'tp', 'dont', 'stop', 'love', 'dance', 'heart', 'không', 'chẳng', 'giờ']
no_diacritics                ['lac', 'troi', 'son', 'tung', 'm', 'tp', 'dont', 'stop', 'loving', 'the', 'dancing', 'hearts', 'khong', 'chang', 'bao', 'gio']
no_stopwords+no_diacritics   ['lac', 'troi', 'son', 'tung', 'tp', 'dont', 'stop', 'loving', 'dancing', 'hearts', 'khong', 'chang', 'gio']


Dùng: `from Dang_preprocess_tokens import preprocess_tokens` (file `.py` đi kèm, cùng code với cell ở trên). Đầu vào là chuỗi đã qua `clean_text`; dùng **cùng strategy cho bài hát và query**.